# 2.1 — Convex Sets & Functions

Convexity is the geometric promise that straight-line mixtures behave well: feasible mixtures stay feasible, and function values along a mixture never rise above the straight chord between endpoint values. In this lesson, you will build convex sets, convex functions, Jensen checks, tangent lower bounds, and curvature certificates from scratch with NumPy so later optimization guarantees feel mechanical rather than magical.

## 📖 Concept walkthrough — build each idea from scratch

Before the worked examples, we build convexity one idea at a time. Run each cell in order and inspect the printed numbers and plots — every inequality is checked directly, and every visualization is chosen to reveal the geometry. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

▶ What you'll see: the only tools used in this lesson are NumPy for arrays and Matplotlib for visual inspection.

### 1. Convex combinations and convex sets

A convex set is a region that contains the whole straight segment between any two of its points. Algebraically, the segment is generated by convex combinations: $z=\lambda x+(1-\lambda)y$ with $0\le\lambda\le1$. The weights are nonnegative and sum to one, so the new point is an average, not an extrapolation.

In [ ]:
x_w = np.array([0.0, 0.0])
y_w = np.array([2.0, 1.0])
lam_w = 0.25
z_w = lam_w * x_w + (1 - lam_w) * y_w
print("x:", x_w, "y:", y_w, "lambda:", lam_w)
print("convex combination z:", z_w)
assert np.allclose(z_w, [1.5, 0.75])

▶ What you'll see: the weighted average lands at `[1.5, 0.75]`, between the two endpoints.

In [ ]:
square_min_w = np.array([0.0, 0.0])
square_max_w = np.array([2.0, 2.0])
inside_square_w = np.all((z_w >= square_min_w) & (z_w <= square_max_w))
print("inside [0,2]^2?", inside_square_w)
assert inside_square_w

▶ What you'll see: the mixture remains inside the square, which is the feasibility-preservation property.

In [ ]:
lams_w = np.linspace(0, 1, 25)
segment_w = np.array([lam * x_w + (1 - lam) * y_w for lam in lams_w])
plt.figure(figsize=(4.4, 3.4))
plt.plot([0, 2, 2, 0, 0], [0, 0, 2, 2, 0], color="black", label="square boundary")
plt.plot(segment_w[:, 0], segment_w[:, 1], color="teal", marker=".", label="all mixtures")
plt.scatter([x_w[0], y_w[0], z_w[0]], [x_w[1], y_w[1], z_w[1]], color=["green", "orange", "red"])
plt.xlim(-0.2, 2.2); plt.ylim(-0.2, 2.2); plt.gca().set_aspect("equal")
plt.title("1: segment stays inside a convex square")
plt.legend(); plt.show()

▶ What you'll see: every dot on the straight segment stays inside the square.

*Why it's done this way:* Convex combinations are the smallest algebraic test for “straight-line safety.” Optimization algorithms repeatedly average, interpolate, and step along lines; if the constraint set is closed under those mixtures, local movement between feasible points cannot accidentally leave the feasible region.

### 2. Nonconvex sets fail by one missing midpoint

To prove a set is not convex, we do not need to check every pair. One counterexample is enough: find two feasible points whose midpoint is infeasible. A ring is the classic example — connected, but with a hole where a straight chord can leave the set.

In [ ]:
p_w = np.array([1.0, 0.0])
q_w = np.array([-1.0, 0.0])
mid_w = 0.5 * p_w + 0.5 * q_w
radius_mid_w = np.linalg.norm(mid_w)
print("p:", p_w, "q:", q_w, "midpoint:", mid_w)
print("midpoint radius:", radius_mid_w)
assert np.allclose(mid_w, [0.0, 0.0])

▶ What you'll see: opposite points on a ring average to the origin.

In [ ]:
inner_w, outer_w = 0.7, 1.3
p_in_ring_w = inner_w <= np.linalg.norm(p_w) <= outer_w
q_in_ring_w = inner_w <= np.linalg.norm(q_w) <= outer_w
mid_in_ring_w = inner_w <= radius_mid_w <= outer_w
print("endpoints in ring?", p_in_ring_w, q_in_ring_w)
print("midpoint in ring?", mid_in_ring_w)
assert p_in_ring_w and q_in_ring_w and not mid_in_ring_w

▶ What you'll see: both endpoints are feasible, but their midpoint falls in the hole.

In [ ]:
theta_w = np.linspace(0, 2 * np.pi, 300)
plt.figure(figsize=(4, 4))
plt.fill(outer_w * np.cos(theta_w), outer_w * np.sin(theta_w), color="lightgray")
plt.fill(inner_w * np.cos(theta_w), inner_w * np.sin(theta_w), color="white")
plt.plot([p_w[0], q_w[0]], [p_w[1], q_w[1]], color="crimson", linewidth=2, label="chord")
plt.scatter([p_w[0], q_w[0], mid_w[0]], [p_w[1], q_w[1], mid_w[1]], color=["green", "green", "red"])
plt.gca().set_aspect("equal"); plt.title("2: a ring is connected but not convex")
plt.legend(); plt.show()

▶ What you'll see: the chord cuts through the empty center, so connectedness is not enough.

*Why it's done this way:* Convexity is a universal statement over all pairs and all weights, so one violating mixture disproves it. The midpoint is the simplest mixture to test because both weights are equal and the geometry is easiest to see.

### 3. Convex functions and Jensen's inequality

A function is convex when the graph lies below every chord between two graph points. The algebraic form is $f(\lambda x+(1-\lambda)y)\le\lambda f(x)+(1-\lambda)f(y)$. For $f(x)=x^2$, this says the square of an average is no larger than the average of the squares.

In [ ]:
x1_w, x2_w = 1.0, 3.0
lam_j_w = 0.25
mix_w = lam_j_w * x1_w + (1 - lam_j_w) * x2_w
left_w = mix_w ** 2
right_w = lam_j_w * x1_w ** 2 + (1 - lam_j_w) * x2_w ** 2
print("mixed input:", mix_w)
print("f(mixed input):", left_w)
print("mixed function values:", right_w)
assert left_w == 6.25 and right_w == 7.0 and left_w <= right_w

▶ What you'll see: `6.25 <= 7.0`, the Jensen inequality for this pair.

In [ ]:
grid_w = np.linspace(0, 4, 200)
fgrid_w = grid_w ** 2
chord_x_w = np.array([x1_w, x2_w])
chord_y_w = chord_x_w ** 2
plt.figure(figsize=(4.4, 3.2))
plt.plot(grid_w, fgrid_w, color="teal", label="f(x)=x²")
plt.plot(chord_x_w, chord_y_w, color="black", linestyle="--", label="chord")
plt.scatter([mix_w], [left_w], color="green", label="f(mix)")
plt.scatter([mix_w], [right_w], color="red", label="chord at mix")
plt.title("3: convex graph sits below the chord")
plt.legend(); plt.show()

▶ What you'll see: the green point on the curve is below the red point on the chord.

*Why it's done this way:* We compare `f` after averaging inputs against averaging the endpoint outputs because optimization constantly replaces hard landscapes with linear interpolation. A convex function rewards averaging: uncertainty or mixing in the input cannot make the function value worse than the corresponding average output.

### 4. Curvature and tangent lower bounds certify convexity

For smooth one-dimensional functions, nonnegative second derivative is a local curvature certificate for convexity. The same geometry says every tangent line is a global lower bound: $f(x)\ge f(x_0)+f'(x_0)(x-x_0)$. That tangent-support property is why gradients later certify global optima in convex optimization.

In [ ]:
x0_w = 2.0
x_test_w = 3.0
f0_w = x0_w ** 2
grad0_w = 2 * x0_w
tangent_at_test_w = f0_w + grad0_w * (x_test_w - x0_w)
f_test_w = x_test_w ** 2
print("tangent at x=3:", tangent_at_test_w)
print("f(3):", f_test_w)
assert tangent_at_test_w == 8.0 and f_test_w == 9.0 and f_test_w >= tangent_at_test_w

▶ What you'll see: the function value 9 sits above the tangent value 8.

In [ ]:
xs_w = np.linspace(-1, 5, 200)
tangent_w = f0_w + grad0_w * (xs_w - x0_w)
plt.figure(figsize=(4.4, 3.2))
plt.plot(xs_w, xs_w ** 2, color="teal", label="f(x)=x²")
plt.plot(xs_w, tangent_w, color="crimson", linestyle="--", label="tangent at x0=2")
plt.scatter([x0_w, x_test_w], [f0_w, f_test_w], color="black")
plt.title("4: tangent is a global lower bound")
plt.legend(); plt.show()

▶ What you'll see: the dashed line touches the bowl at `x0=2` and stays below it everywhere shown.

In [ ]:
H_w = np.array([[2.0, 0.0], [0.0, 6.0]])
eigs_w = np.linalg.eigvalsh(H_w)
print("Hessian eigenvalues:", eigs_w)
print("positive semidefinite?", np.all(eigs_w >= 0))
assert np.allclose(eigs_w, [2.0, 6.0])

▶ What you'll see: both Hessian eigenvalues are positive, so the quadratic bowl curves upward in every direction.

*Why it's done this way:* A second derivative or Hessian checks curvature locally, while the tangent inequality turns that local curvature into a global support statement. Positive Hessian eigenvalues mean every direction has upward curvature, so no direction can hide a local valley that is not globally consistent with the bowl.

## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 🟢 Basics (warm-up)

### Basic 1 — Build a convex combination

**Goal.** Average two vectors with nonnegative weights that sum to one, because convexity is about straight-line mixtures.

In [ ]:
x_b1 = np.array([0.0, 0.0])
y_b1 = np.array([2.0, 1.0])
lam_b1 = 0.25
z_b1 = lam_b1 * x_b1 + (1 - lam_b1) * y_b1
print("z_b1:", z_b1)
assert np.allclose(z_b1, [1.5, 0.75])

▶ What you'll see: the combination is a point between the endpoints.

👀 Takeaway: convex combinations are weighted averages, not arbitrary linear combinations.

### Basic 2 — Check that a square is convex for one segment

**Goal.** Verify that a segment between two points in `[0,2]^2` stays inside the square.

In [ ]:
x_b2 = np.array([0.2, 1.8])
y_b2 = np.array([1.8, 0.4])
lams_b2 = np.linspace(0, 1, 11)
pts_b2 = np.array([lam * x_b2 + (1 - lam) * y_b2 for lam in lams_b2])
inside_b2 = np.all((pts_b2 >= 0) & (pts_b2 <= 2))
print("sampled points:\n", np.round(pts_b2, 2))
print("all inside square?", inside_b2)
assert inside_b2

▶ What you'll see: every sampled point has both coordinates between 0 and 2.

👀 Takeaway: boxes are convex because coordinate-wise averages remain between coordinate-wise bounds.

### Basic 3 — See a nonconvex midpoint failure

**Goal.** Use two points in a ring to show that connected does not imply convex.

In [ ]:
p_b3 = np.array([1.0, 0.0])
q_b3 = np.array([-1.0, 0.0])
mid_b3 = 0.5 * (p_b3 + q_b3)
inner_b3, outer_b3 = 0.7, 1.3
mid_ok_b3 = inner_b3 <= np.linalg.norm(mid_b3) <= outer_b3
print("midpoint:", mid_b3, "in ring?", mid_ok_b3)
assert not mid_ok_b3

▶ What you'll see: the midpoint is the origin, which lies in the ring's hole.

👀 Takeaway: one infeasible midpoint is enough to disprove set convexity.

### Basic 4 — Plot a convex segment inside a disk

**Goal.** Visualize that the disk contains the chord between two interior points.

In [ ]:
x_b4 = np.array([0.6, 0.2])
y_b4 = np.array([-0.3, 0.7])
lams_b4 = np.linspace(0, 1, 30)
seg_b4 = np.array([lam * x_b4 + (1 - lam) * y_b4 for lam in lams_b4])
radii_b4 = np.linalg.norm(seg_b4, axis=1)
print("max segment radius:", round(float(radii_b4.max()), 3))
assert radii_b4.max() <= 1.0
plt.figure(figsize=(4, 4))
t_b4 = np.linspace(0, 2 * np.pi, 200)
plt.plot(np.cos(t_b4), np.sin(t_b4), color="black")
plt.plot(seg_b4[:, 0], seg_b4[:, 1], color="teal")
plt.scatter([x_b4[0], y_b4[0]], [x_b4[1], y_b4[1]], color="orange")
plt.gca().set_aspect("equal"); plt.title("Basic 4: segment in unit disk"); plt.show()

▶ What you'll see: the segment lies completely inside the circular boundary.

👀 Takeaway: Euclidean balls are convex because averaging cannot increase distance beyond the endpoint bound.

### Basic 5 — Check Jensen for a square function

**Goal.** Compare `f` at an averaged input with the averaged function values.

In [ ]:
x_b5, y_b5, lam_b5 = 1.0, 3.0, 0.25
mix_b5 = lam_b5 * x_b5 + (1 - lam_b5) * y_b5
lhs_b5 = mix_b5 ** 2
rhs_b5 = lam_b5 * x_b5 ** 2 + (1 - lam_b5) * y_b5 ** 2
print("f(mix):", lhs_b5, "weighted f values:", rhs_b5)
assert lhs_b5 == 6.25 and rhs_b5 == 7.0 and lhs_b5 <= rhs_b5

▶ What you'll see: the convexity inequality holds numerically.

👀 Takeaway: convex functions put the function value at an average below the chord value.

### Basic 6 — Plot the chord above `x²`

**Goal.** Make the Jensen inequality visible on a graph.

In [ ]:
x_b6 = np.linspace(0, 4, 200)
y_b6 = x_b6 ** 2
end_x_b6 = np.array([1.0, 3.0])
end_y_b6 = end_x_b6 ** 2
plt.figure(figsize=(4.4, 3.2))
plt.plot(x_b6, y_b6, label="x²", color="teal")
plt.plot(end_x_b6, end_y_b6, "--", label="chord", color="black")
plt.scatter(end_x_b6, end_y_b6, color="orange")
plt.title("Basic 6: chord above a convex graph")
plt.legend(); plt.show()

▶ What you'll see: the dashed chord stays above the curved graph between the endpoints.

👀 Takeaway: chord-above-graph is the geometric definition of convex functions.

### Basic 7 — Use a second derivative certificate

**Goal.** Confirm that `f(x)=x²` is convex by checking its curvature.

In [ ]:
xs_b7 = np.array([-2.0, 0.0, 2.0])
second_derivative_b7 = np.full_like(xs_b7, 2.0)
print("sample x values:", xs_b7)
print("f''(x):", second_derivative_b7)
assert np.all(second_derivative_b7 >= 0)

▶ What you'll see: the second derivative is positive at every sampled location.

👀 Takeaway: in one dimension, nonnegative second derivative is a smooth convexity certificate.

### Basic 8 — Check a tangent lower bound

**Goal.** Verify that a tangent line to a convex function stays below the function.

In [ ]:
x0_b8 = 2.0
x_b8 = np.array([1.0, 2.0, 3.0])
f_b8 = x_b8 ** 2
tangent_b8 = x0_b8 ** 2 + 2 * x0_b8 * (x_b8 - x0_b8)
print("f values:", f_b8)
print("tangent values:", tangent_b8)
assert np.all(f_b8 >= tangent_b8)

▶ What you'll see: the tangent matches at `x=2` and is lower at neighboring points.

👀 Takeaway: tangent lower bounds are the gradient-based signature of convexity.

### Basic 9 — Recognize a convex nonsmooth function

**Goal.** Inspect `|x|`, which is convex even though it has a corner at zero.

In [ ]:
x_b9 = np.linspace(-2, 2, 9)
f_b9 = np.abs(x_b9)
mid_b9 = 0.5 * (-1.0) + 0.5 * (1.0)
lhs_b9 = abs(mid_b9)
rhs_b9 = 0.5 * abs(-1.0) + 0.5 * abs(1.0)
print("|x| values:", f_b9)
print("midpoint Jensen check:", lhs_b9, "<=", rhs_b9)
assert lhs_b9 <= rhs_b9

▶ What you'll see: the V-shape satisfies Jensen even at the nondifferentiable corner.

👀 Takeaway: convexity does not require smoothness; corners can still be globally bowl-shaped.

### Basic 10 — Check a Hessian's eigenvalues

**Goal.** Use positive eigenvalues to certify a two-dimensional quadratic bowl.

In [ ]:
H_b10 = np.array([[2.0, 0.0], [0.0, 6.0]])
eigs_b10 = np.linalg.eigvalsh(H_b10)
print("Hessian eigenvalues:", eigs_b10)
assert np.allclose(eigs_b10, [2.0, 6.0])
assert np.all(eigs_b10 >= 0)

▶ What you'll see: every curvature direction is nonnegative.

👀 Takeaway: a symmetric Hessian with nonnegative eigenvalues means a smooth quadratic is convex.

## 🟡 Easy

### Easy 1 — Test many mixtures in a triangle

**Goal.** Sample convex combinations of triangle vertices, because a simplex is the basic convex hull of finitely many points.

In [ ]:
V_e1 = np.array([[0.0, 0.0], [2.0, 0.0], [0.5, 1.5]])
weights_e1 = np.array([[0.2, 0.3, 0.5], [0.6, 0.1, 0.3], [0.1, 0.8, 0.1]])
pts_e1 = weights_e1 @ V_e1
print("weights row sums:", weights_e1.sum(axis=1))
print("triangle mixtures:\n", np.round(pts_e1, 3))
assert np.allclose(weights_e1.sum(axis=1), 1.0)

▶ What you'll see: each row of weights creates one point inside the triangle.

In [ ]:
plt.figure(figsize=(4.4, 3.4))
closed_e1 = np.vstack([V_e1, V_e1[0]])
plt.plot(closed_e1[:, 0], closed_e1[:, 1], color="black")
plt.scatter(pts_e1[:, 0], pts_e1[:, 1], color="teal")
plt.gca().set_aspect("equal"); plt.title("Easy 1: convex hull of three vertices"); plt.show()

▶ What you'll see: all sampled mixtures lie inside the triangular hull.

👀 Takeaway: convex hull points are exactly nonnegative weighted averages whose weights sum to one.

### Easy 2 — Show a halfspace is convex

**Goal.** Verify that if two points satisfy `a·x <= b`, then their average does too.

In [ ]:
a_e2 = np.array([1.0, 2.0])
b_e2 = 5.0
x_e2 = np.array([1.0, 1.0])
y_e2 = np.array([3.0, 0.5])
lam_e2 = 0.4
z_e2 = lam_e2 * x_e2 + (1 - lam_e2) * y_e2
values_e2 = np.array([a_e2 @ x_e2, a_e2 @ y_e2, a_e2 @ z_e2])
print("a·x, a·y, a·z:", values_e2)
assert np.all(values_e2 <= b_e2)

▶ What you'll see: the mixed point still satisfies the linear inequality.

In [ ]:
xx_e2 = np.linspace(0, 5, 100)
yline_e2 = (b_e2 - xx_e2) / 2
plt.figure(figsize=(4.4, 3.2))
plt.plot(xx_e2, yline_e2, color="black", label="a·x=b")
plt.fill_between(xx_e2, -0.2, yline_e2, color="lightblue", alpha=0.5, label="a·x<=b")
plt.scatter([x_e2[0], y_e2[0], z_e2[0]], [x_e2[1], y_e2[1], z_e2[1]], color=["green", "orange", "red"])
plt.ylim(-0.2, 3); plt.title("Easy 2: halfspace mixture"); plt.legend(); plt.show()

▶ What you'll see: the average point stays in the shaded feasible halfspace.

👀 Takeaway: linear inequalities define convex feasible regions because dot products preserve weighted averages.

### Easy 3 — Compare convex and concave chord tests

**Goal.** See that `x²` passes the chord test while `-x²` fails it for minimization.

In [ ]:
x_e3, y_e3, lam_e3 = -1.0, 2.0, 0.5
mix_e3 = lam_e3 * x_e3 + (1 - lam_e3) * y_e3
convex_lhs_e3 = mix_e3 ** 2
convex_rhs_e3 = lam_e3 * x_e3 ** 2 + (1 - lam_e3) * y_e3 ** 2
concave_lhs_e3 = -mix_e3 ** 2
concave_rhs_e3 = lam_e3 * (-x_e3 ** 2) + (1 - lam_e3) * (-y_e3 ** 2)
print("x² check:", convex_lhs_e3, "<=", convex_rhs_e3)
print("-x² check:", concave_lhs_e3, "<=", concave_rhs_e3)
assert convex_lhs_e3 <= convex_rhs_e3 and not (concave_lhs_e3 <= concave_rhs_e3)

▶ What you'll see: the bowl passes, but the upside-down bowl violates the convex inequality.

In [ ]:
grid_e3 = np.linspace(-2, 2, 200)
plt.figure(figsize=(4.4, 3.2))
plt.plot(grid_e3, grid_e3 ** 2, label="convex x²", color="teal")
plt.plot(grid_e3, -grid_e3 ** 2, label="nonconvex -x²", color="crimson")
plt.title("Easy 3: bowl versus upside-down bowl")
plt.legend(); plt.show()

▶ What you'll see: the convex bowl opens upward, while the concave curve opens downward.

👀 Takeaway: convexity is directional; `-x²` is good for maximization but nonconvex for minimization.

### Easy 4 — Verify Jensen for a weighted expectation

**Goal.** Treat weights as probabilities and check Jensen's inequality for `exp(x)`.

In [ ]:
values_e4 = np.array([-1.0, 0.0, 2.0])
probs_e4 = np.array([0.2, 0.5, 0.3])
mean_e4 = float(np.sum(probs_e4 * values_e4))
lhs_e4 = np.exp(mean_e4)
rhs_e4 = float(np.sum(probs_e4 * np.exp(values_e4)))
print("E[X]:", round(mean_e4, 3))
print("exp(E[X]):", round(lhs_e4, 3), "E[exp(X)]:", round(rhs_e4, 3))
assert lhs_e4 <= rhs_e4

▶ What you'll see: the exponential at the mean is below the probability-weighted exponential values.

In [ ]:
plt.figure(figsize=(4.4, 3.2))
plt.bar(["exp(E[X])", "E[exp(X)]"], [lhs_e4, rhs_e4], color=["teal", "orange"])
plt.title("Easy 4: Jensen as weighted averaging")
plt.ylabel("value"); plt.show()

▶ What you'll see: the Jensen upper bar is taller.

👀 Takeaway: expectations are convex combinations, so Jensen is convexity applied to probability weights.

### Easy 5 — Draw tangent lower bounds for a quadratic

**Goal.** Plot several tangent lines to see that each supports the same convex function from below.

In [ ]:
xs_e5 = np.linspace(-3, 3, 200)
f_e5 = xs_e5 ** 2
anchors_e5 = np.array([-2.0, 0.0, 2.0])
tangents_e5 = np.array([a ** 2 + 2 * a * (xs_e5 - a) for a in anchors_e5])
violations_e5 = np.max(tangents_e5 - f_e5)
print("largest tangent minus function:", round(float(violations_e5), 8))
assert violations_e5 <= 1e-10

▶ What you'll see: no tangent line rises above the quadratic.

In [ ]:
plt.figure(figsize=(4.6, 3.2))
plt.plot(xs_e5, f_e5, color="black", linewidth=2, label="x²")
for i_e5, a_e5 in enumerate(anchors_e5):
    plt.plot(xs_e5, tangents_e5[i_e5], linestyle="--", label=f"tangent {a_e5:g}")
plt.ylim(-2, 10); plt.title("Easy 5: tangent lower bounds")
plt.legend(); plt.show()

▶ What you'll see: every dashed tangent touches once and stays below the bowl.

👀 Takeaway: convex gradients define global lower bounds, not just local slopes.

## 🔴 Advanced

### Advanced 1 — Compare local minima in convex and nonconvex landscapes

**Goal.** Sample two one-dimensional functions to see why convexity rules out deceptive local minima.

In [ ]:
x_a1 = np.linspace(-3, 3, 601)
convex_a1 = (x_a1 - 0.5) ** 2
nonconvex_a1 = (x_a1 ** 2 - 1) ** 2 + 0.15 * x_a1
convex_min_x_a1 = x_a1[np.argmin(convex_a1)]
nonconvex_min_x_a1 = x_a1[np.argmin(nonconvex_a1)]
print("convex grid minimizer:", round(float(convex_min_x_a1), 3))
print("nonconvex best grid minimizer:", round(float(nonconvex_min_x_a1), 3))
assert abs(convex_min_x_a1 - 0.5) < 0.01

▶ What you'll see: the convex quadratic has one bottom, while the nonconvex curve has multiple valleys.

In [ ]:
plt.figure(figsize=(5, 3.2))
plt.plot(x_a1, convex_a1, label="convex", color="teal")
plt.plot(x_a1, nonconvex_a1, label="nonconvex", color="crimson")
plt.title("Advanced 1: convex bowl vs nonconvex valleys")
plt.legend(); plt.show()

▶ What you'll see: the red curve has separate basins, which can trap local methods.

👀 Takeaway: convex minimization is trustworthy because any local minimum is global.

### Advanced 2 — Certify a quadratic with Hessian eigenvalues

**Goal.** Build a quadratic form and classify it from its Hessian spectrum.

In [ ]:
H_a2 = np.array([[4.0, 1.0], [1.0, 3.0]])
b_a2 = np.array([1.0, -2.0])
eigs_a2 = np.linalg.eigvalsh(H_a2)
x_star_a2 = -np.linalg.solve(H_a2, b_a2)
print("H eigenvalues:", np.round(eigs_a2, 3))
print("stationary point:", np.round(x_star_a2, 3))
assert np.all(eigs_a2 > 0)

▶ What you'll see: positive eigenvalues certify a strictly convex quadratic with one stationary point.

In [ ]:
xg_a2 = np.linspace(-1.5, 1.5, 80)
yg_a2 = np.linspace(-1.5, 1.5, 80)
X_a2, Y_a2 = np.meshgrid(xg_a2, yg_a2)
Z_a2 = 0.5 * (H_a2[0, 0] * X_a2 ** 2 + 2 * H_a2[0, 1] * X_a2 * Y_a2 + H_a2[1, 1] * Y_a2 ** 2) + b_a2[0] * X_a2 + b_a2[1] * Y_a2
plt.figure(figsize=(4.6, 3.6))
plt.contour(X_a2, Y_a2, Z_a2, levels=18, cmap="viridis")
plt.scatter([x_star_a2[0]], [x_star_a2[1]], color="red", label="global minimizer")
plt.title("Advanced 2: convex quadratic contours")
plt.legend(); plt.show()

▶ What you'll see: elliptical contours surround the unique minimizer.

👀 Takeaway: positive definite Hessians turn stationarity into a global minimum for quadratics.

### Advanced 3 — Estimate curvature with finite differences

**Goal.** Approximate second derivatives numerically, because real objectives may be available only as function evaluations.

In [ ]:
def f_a3(x_a3):
    return np.log1p(np.exp(x_a3))

xs_a3 = np.linspace(-4, 4, 41)
h_a3 = 1e-2
second_fd_a3 = (f_a3(xs_a3 + h_a3) - 2 * f_a3(xs_a3) + f_a3(xs_a3 - h_a3)) / h_a3 ** 2
print("min finite-difference curvature:", round(float(second_fd_a3.min()), 6))
print("max finite-difference curvature:", round(float(second_fd_a3.max()), 6))
assert second_fd_a3.min() > 0

▶ What you'll see: the softplus function has positive estimated curvature everywhere sampled.

In [ ]:
plt.figure(figsize=(4.6, 3.2))
plt.plot(xs_a3, second_fd_a3, color="purple")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Advanced 3: numerical curvature of softplus")
plt.xlabel("x"); plt.ylabel("finite-difference f''(x)"); plt.show()

▶ What you'll see: the curvature curve stays above zero.

👀 Takeaway: sampled curvature checks do not prove convexity globally, but they are useful diagnostics for smooth losses.

### Advanced 4 — Project a point onto a convex set

**Goal.** Project onto the probability simplex, because constrained convex optimization often repeatedly returns points to a convex feasible set.

In [ ]:
v_a4 = np.array([0.7, -0.2, 1.5])
u_a4 = np.sort(v_a4)[::-1]
cssv_a4 = np.cumsum(u_a4) - 1
rho_candidates_a4 = u_a4 - cssv_a4 / (np.arange(len(v_a4)) + 1) > 0
rho_a4 = np.where(rho_candidates_a4)[0][-1]
theta_a4 = cssv_a4[rho_a4] / (rho_a4 + 1)
w_a4 = np.maximum(v_a4 - theta_a4, 0)
print("theta:", round(float(theta_a4), 3))
print("projected point:", np.round(w_a4, 3), "sum:", round(float(w_a4.sum()), 3))
assert np.all(w_a4 >= 0) and np.allclose(w_a4.sum(), 1.0)

▶ What you'll see: the infeasible vector becomes a nonnegative vector summing to one.

In [ ]:
plt.figure(figsize=(4.6, 3.0))
idx_a4 = np.arange(len(v_a4))
plt.bar(idx_a4 - 0.18, v_a4, width=0.36, label="original", color="crimson")
plt.bar(idx_a4 + 0.18, w_a4, width=0.36, label="projected", color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Advanced 4: projection onto simplex")
plt.legend(); plt.show()

▶ What you'll see: negative mass is clipped and excess mass is redistributed to satisfy the simplex constraint.

👀 Takeaway: projections rely on convexity so the closest feasible point is well-defined and unique for squared distance.

### Advanced 5 — Separate a point from a convex ball with a tangent hyperplane

**Goal.** Construct a supporting line to the unit ball, because convex sets can be certified by linear inequalities at their boundary.

In [ ]:
boundary_a5 = np.array([1.0 / np.sqrt(2), 1.0 / np.sqrt(2)])
outside_a5 = np.array([1.2, 1.2])
normal_a5 = boundary_a5.copy()
boundary_value_a5 = normal_a5 @ boundary_a5
outside_value_a5 = normal_a5 @ outside_a5
print("support value at boundary:", round(float(boundary_value_a5), 3))
print("outside value:", round(float(outside_value_a5), 3))
assert np.allclose(boundary_value_a5, 1.0) and outside_value_a5 > 1.0

▶ What you'll see: the outside point violates the tangent halfspace inequality while the boundary point is tight.

In [ ]:
t_a5 = np.linspace(0, 2 * np.pi, 300)
line_x_a5 = np.linspace(-0.2, 1.4, 100)
line_y_a5 = (1 - normal_a5[0] * line_x_a5) / normal_a5[1]
plt.figure(figsize=(4.4, 4.0))
plt.plot(np.cos(t_a5), np.sin(t_a5), color="black", label="unit ball boundary")
plt.plot(line_x_a5, line_y_a5, "--", color="crimson", label="supporting line")
plt.scatter([boundary_a5[0], outside_a5[0]], [boundary_a5[1], outside_a5[1]], color=["green", "red"])
plt.gca().set_aspect("equal"); plt.xlim(-1.2, 1.5); plt.ylim(-1.2, 1.5)
plt.title("Advanced 5: tangent separates outside point")
plt.legend(); plt.show()

▶ What you'll see: the dashed line touches the ball once and leaves the outside point on the wrong side.

👀 Takeaway: supporting hyperplanes turn convex geometry into linear certificates.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Convexity is the promise that straight-line mixtures behave: feasible mixtures stay feasible, and function chords stay above the graph.

Convexity is the language that makes optimization trustworthy. Weighted averages, Jensen's inequality, and curvature certificates are why later optimizers can distinguish reliable bowls from deceptive shapes. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

SEED = 20260701
rng = np.random.default_rng(SEED)


def quadratic_surface(name, H, center, x0, project=None):
    H = np.asarray(H, dtype=float)
    center = np.asarray(center, dtype=float)
    x0 = np.asarray(x0, dtype=float)

    def loss(x):
        z = np.asarray(x, dtype=float) - center
        return 0.5 * float(z @ H @ z)

    def grad(x):
        z = np.asarray(x, dtype=float) - center
        return H @ z

    def hess(x):
        return H

    return {
        "name": name,
        "kind": "quadratic",
        "dim": len(x0),
        "x0": x0,
        "loss": loss,
        "grad": grad,
        "hess": hess,
        "project": project,
        "center": center,
    }


def rosenbrock_surface():
    def loss(x):
        x = np.asarray(x, dtype=float)
        a = x[0]
        b = x[1]
        return float(100.0 * (b - a * a) ** 2 + (1.0 - a) ** 2)

    def grad(x):
        x = np.asarray(x, dtype=float)
        a = x[0]
        b = x[1]
        return np.array([
            -400.0 * a * (b - a * a) - 2.0 * (1.0 - a),
            200.0 * (b - a * a),
        ])

    def hess(x):
        x = np.asarray(x, dtype=float)
        a = x[0]
        b = x[1]
        return np.array([
            [1200.0 * a * a - 400.0 * b + 2.0, -400.0 * a],
            [-400.0 * a, 200.0],
        ])

    return {
        "name": "D3 nonconvex Rosenbrock valley",
        "kind": "rosenbrock",
        "dim": 2,
        "x0": np.array([-1.2, 1.0]),
        "loss": loss,
        "grad": grad,
        "hess": hess,
        "project": None,
        "center": np.array([1.0, 1.0]),
    }


def logistic_surface():
    data = load_breast_cancer()
    X = StandardScaler().fit_transform(data.data)
    X = np.column_stack([np.ones(X.shape[0]), X])
    y = data.target.astype(float)
    reg = 0.05

    def loss(w):
        z = X @ w
        yz = y * z
        logistic = np.logaddexp(0.0, z) - yz
        penalty = 0.5 * reg * float(w[1:] @ w[1:])
        return float(np.mean(logistic) + penalty)

    def grad(w):
        z = X @ w
        p = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
        g = X.T @ (p - y) / X.shape[0]
        g[1:] = g[1:] + reg * w[1:]
        return g

    def hess(w):
        z = X @ w
        p = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
        weights = p * (1.0 - p)
        Xw = X * weights[:, None]
        H = X.T @ Xw / X.shape[0]
        H[1:, 1:] = H[1:, 1:] + reg * np.eye(X.shape[1] - 1)
        return H

    return {
        "name": "D4 breast-cancer logistic loss",
        "kind": "logistic",
        "dim": X.shape[1],
        "x0": np.zeros(X.shape[1]),
        "loss": loss,
        "grad": grad,
        "hess": hess,
        "project": None,
        "center": np.zeros(X.shape[1]),
        "X_shape": X.shape,
        "classes": sorted(set(data.target.tolist())),
    }


def make_high_dim_box_surface(dim=20):
    Q, _ = np.linalg.qr(rng.normal(size=(dim, dim)))
    eigs = np.geomspace(1.0, 120.0, dim)
    H = Q @ np.diag(eigs) @ Q.T
    center = np.linspace(-0.8, 0.8, dim)
    x0 = np.linspace(1.4, -1.4, dim)

    def project(x):
        return np.clip(x, -1.0, 1.0)

    return quadratic_surface("D5 high-dimensional box-constrained SPD bowl", H, center, x0, project=project)


def make_loss_ladder():
    d1 = quadratic_surface(
        "D1 2-D quadratic bowl",
        np.diag([2.0, 4.0]),
        np.array([1.0, -1.0]),
        np.array([-2.0, 2.0]),
    )
    d2 = quadratic_surface(
        "D2 anisotropic ill-conditioned quadratic",
        np.diag([1.0, 80.0]),
        np.array([-1.0, 1.0]),
        np.array([2.5, -2.0]),
    )
    return [d1, d2, rosenbrock_surface(), logistic_surface(), make_high_dim_box_surface()]


def project_if_needed(surface, x):
    if surface.get("project") is None:
        return np.asarray(x, dtype=float)
    return surface["project"](np.asarray(x, dtype=float))


def run_optimizer(surface, steps=120, eta=0.05, tol=1e-6):
    x = project_if_needed(surface, surface["x0"])
    losses = []
    path = [x.copy()]
    for step in range(steps):
        loss_value = surface["loss"](x)
        losses.append(loss_value)
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        trial = x - eta * g
        x = project_if_needed(surface, trial)
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def backtracking_line_search(loss, grad, x, direction, alpha0=1.0, rho=0.5, c=0.5):
    alpha = alpha0
    f0 = loss(x)
    g0 = grad(x)
    slope = float(g0 @ direction)
    while loss(x + alpha * direction) > f0 + c * alpha * slope:
        alpha = rho * alpha
        if alpha < 1e-10:
            break
    return alpha


def line_search_optimizer(surface, steps=80, tol=1e-6):
    x = project_if_needed(surface, surface["x0"])
    losses = []
    path = [x.copy()]
    for step in range(steps):
        losses.append(surface["loss"](x))
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        direction = -g
        alpha = backtracking_line_search(surface["loss"], surface["grad"], x, direction)
        x = project_if_needed(surface, x + alpha * direction)
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def bfgs_update(B, s, y):
    Bs = B @ s
    sBs = float(s @ Bs)
    ys = float(y @ s)
    if ys <= 1e-12 or sBs <= 1e-12:
        return B
    return B - np.outer(Bs, Bs) / sBs + np.outer(y, y) / ys


def damped_newton_optimizer(surface, steps=50, tol=1e-6, damping=1e-4):
    x = project_if_needed(surface, surface["x0"])
    losses = []
    path = [x.copy()]
    for step in range(steps):
        losses.append(surface["loss"](x))
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        H = surface["hess"](x)
        shift = damping
        eye = np.eye(len(x))
        while np.min(np.linalg.eigvalsh(H + shift * eye)) <= 0.0:
            shift = 10.0 * shift
        direction = -np.linalg.solve(H + shift * eye, g)
        if float(direction @ g) >= 0.0:
            direction = -g
        alpha = backtracking_line_search(surface["loss"], surface["grad"], x, direction, c=1e-4)
        x = project_if_needed(surface, x + alpha * direction)
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def bfgs_optimizer(surface, steps=60, tol=1e-6):
    x = project_if_needed(surface, surface["x0"])
    dim = len(x)
    B = np.eye(dim)
    losses = []
    path = [x.copy()]
    for step in range(steps):
        losses.append(surface["loss"](x))
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        direction = -np.linalg.solve(B + 1e-8 * np.eye(dim), g)
        if float(direction @ g) >= 0.0:
            direction = -g
        alpha = backtracking_line_search(surface["loss"], surface["grad"], x, direction, c=1e-4)
        x_next = project_if_needed(surface, x + alpha * direction)
        s = x_next - x
        y = surface["grad"](x_next) - g
        B = bfgs_update(B, s, y)
        x = x_next
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def conjugate_gradient(A, b, x0=None, tol=1e-8, max_iter=None):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    if x0 is None:
        x = np.zeros_like(b)
    else:
        x = np.asarray(x0, dtype=float).copy()
    if max_iter is None:
        max_iter = len(b)
    r = b - A @ x
    p = r.copy()
    rs_old = float(r @ r)
    residuals = [np.sqrt(rs_old)]
    path = [x.copy()]
    for k in range(max_iter):
        Ap = A @ p
        denom = float(p @ Ap)
        if denom <= 0.0:
            raise ValueError("CG requires symmetric positive-definite curvature")
        alpha = rs_old / denom
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = float(r @ r)
        residuals.append(np.sqrt(rs_new))
        path.append(x.copy())
        if np.sqrt(rs_new) < tol:
            break
        beta = rs_new / rs_old
        p = r + beta * p
        rs_old = rs_new
    return {
        "x": x,
        "residuals": np.array(residuals),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def make_cg_systems():
    systems = []
    base = make_loss_ladder()
    for surface in base:
        if surface["kind"] == "rosenbrock":
            point = np.array([1.0, 1.0])
            A = surface["hess"](point) + 1e-3 * np.eye(2)
            b = A @ point
            name = "D3 Rosenbrock SPD local Newton system"
        elif surface["kind"] == "logistic":
            point = surface["x0"]
            A = surface["hess"](point)
            b = -surface["grad"](point)
            name = "D4 logistic Hessian-vector SPD system"
        else:
            A = surface["hess"](surface["x0"])
            b = A @ surface["center"]
            name = surface["name"].replace("loss", "system")
        systems.append({
            "name": name,
            "A": A,
            "b": b,
            "x0": np.zeros_like(b),
        })
    return systems


def preview_ladder(ladder):
    for idx, item in enumerate(ladder, start=1):
        if "A" in item:
            shape = item["A"].shape
            extra = f"system_shape={shape}"
            sample = item["b"][: min(3, len(item["b"]))]
        else:
            shape = (item["dim"],)
            extra = f"dim={item['dim']} kind={item['kind']}"
            sample = item["x0"][: min(3, len(item["x0"]))]
        print(f"D{idx}: {item['name']} | {extra} | sample={sample}")


def plot_loss_contours(ax, surface, path, title):
    path = np.asarray(path)
    if path.ndim == 1:
        path = path[:, None]
    if path.shape[1] == 1:
        xs = np.linspace(-3.0, 3.0, 120)
        ys = np.zeros_like(xs)
        vals = np.array([surface["loss"](np.array([x])) for x in xs])
        ax.plot(xs, vals)
        ax.set_title(title)
        return
    x_min = min(np.min(path[:, 0]) - 0.5, -2.5)
    x_max = max(np.max(path[:, 0]) + 0.5, 2.5)
    y_min = min(np.min(path[:, 1]) - 0.5, -2.5)
    y_max = max(np.max(path[:, 1]) + 0.5, 2.5)
    xs = np.linspace(x_min, x_max, 100)
    ys = np.linspace(y_min, y_max, 100)
    xx, yy = np.meshgrid(xs, ys)
    base = path[-1].copy()
    zz = np.zeros_like(xx)
    for i in range(xx.shape[0]):
        for j in range(xx.shape[1]):
            point = base.copy()
            point[0] = xx[i, j]
            point[1] = yy[i, j]
            zz[i, j] = surface["loss"](point)
    ax.contour(xx, yy, zz, levels=20)
    ax.plot(path[:, 0], path[:, 1], marker="o", markersize=2)
    ax.set_title(title)


def plot_system_contours(ax, system, path, title):
    A = system["A"]
    b = system["b"]
    path = np.asarray(path)
    if path.shape[1] < 2:
        ax.plot(np.arange(len(path)), path[:, 0])
        ax.set_title(title)
        return
    x_min = min(np.min(path[:, 0]) - 0.5, -2.0)
    x_max = max(np.max(path[:, 0]) + 0.5, 2.0)
    y_min = min(np.min(path[:, 1]) - 0.5, -2.0)
    y_max = max(np.max(path[:, 1]) + 0.5, 2.0)
    xs = np.linspace(x_min, x_max, 90)
    ys = np.linspace(y_min, y_max, 90)
    xx, yy = np.meshgrid(xs, ys)
    base = path[-1].copy()
    zz = np.zeros_like(xx)
    for i in range(xx.shape[0]):
        for j in range(xx.shape[1]):
            point = base.copy()
            point[0] = xx[i, j]
            point[1] = yy[i, j]
            zz[i, j] = 0.5 * float(point @ A @ point) - float(b @ point)
    ax.contour(xx, yy, zz, levels=20)
    ax.plot(path[:, 0], path[:, 1], marker="o", markersize=2)
    ax.set_title(title)

## The concept, built once (D1)

A set is convex when $\lambda x+(1-\lambda)y\in C$ and a function is convex when $f(\lambda x+(1-\lambda)y)\le\lambda f(x)+(1-\lambda)f(y)$.

The D1 check is intentionally tiny: it plugs in the lesson's exact numbers before the same optimizer machinery is used on harder rungs.

In [ ]:
def is_convex_combo(x, y, lam):
    return lam * np.asarray(x, dtype=float) + (1.0 - lam) * np.asarray(y, dtype=float)

x = np.array([0.0, 0.0])
y = np.array([2.0, 1.0])
combo = is_convex_combo(x, y, 0.25)
assert np.allclose(combo, np.array([1.5, 0.75]))
print("convex combination:", combo)

Now connect the geometric test to a reusable optimizer on the D1 bowl. The same `run_optimizer()` will be used on every loss surface; D1 has a closed-form minimum for verification.

In [ ]:
left = 0.25 * 1.0 + 0.75 * 3.0
jensen_left = left ** 2
jensen_right = 0.25 * 1.0 ** 2 + 0.75 * 3.0 ** 2
assert np.isclose(left, 2.5)
assert np.isclose(jensen_left, 6.25)
assert np.isclose(jensen_right, 7.0)
assert jensen_left <= jensen_right

d1 = make_loss_ladder()[0]
result = run_optimizer(d1, eta=0.1, steps=80)
closed_form_min = d1["center"]
print("Jensen:", jensen_left, "<=", jensen_right)
print("D1 final x:", result["x"])
print("closed-form min:", closed_form_min)

## Dataset ladder: D1→D5 loss surfaces

Inline F4 ladder, no shared helper import: D1 quadratic bowl, D2 ill-conditioned quadratic, D3 Rosenbrock/nonconvex, D4 real `load_breast_cancer` logistic loss, and D5 high-dimensional or constrained. The metric is final loss.

In [ ]:
ladder = make_loss_ladder()
preview_ladder(ladder)

## Run the same method across D1→D5

Only the method for this lesson changes; the ladder stays fixed. Collect one metric per rung: final loss.

In [ ]:
ladder = make_loss_ladder()
results = []
for rung, surface in enumerate(ladder, start=1):
    eta = 0.02 if rung in [2, 3, 5] else 0.05
    if surface["kind"] == "logistic":
        eta = 0.2
    result = run_optimizer(surface, eta=eta, steps=160)
    results.append(result)
    final_loss = result["losses"][-1]
    print(f"D{rung} | final_loss={final_loss:.6f} | iterations={result['iterations']}")

## Results visualization

The closing figure has two parts: optimizer trajectories on contour panels, then the metric curve across D1→D5.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for idx, (surface, result) in enumerate(zip(ladder, results)):
    plot_loss_contours(axes[idx], surface, result["path"], f"D{idx + 1}")
plt.tight_layout()
plt.show()

metrics = [result["losses"][-1] for result in results]
plt.figure(figsize=(6, 3.5))
plt.plot(range(1, 6), metrics, marker="o")
plt.xlabel("rung")
plt.ylabel("final loss")
plt.title("Final loss across the ladder")
plt.grid(True)
plt.show()

## Pitfall on the hardest rung

Pitfall on D5: connected is not convex. An annulus is one connected region, but opposite feasible points have an infeasible midpoint in the hole. The fix is to project to a convex set such as a box or simplex before trusting interpolation.

In [ ]:
def in_annulus(point, inner=0.8, outer=1.2):
    radius = np.linalg.norm(point)
    return inner <= radius <= outer

point_a = np.array([1.0, 0.0])
point_b = np.array([-1.0, 0.0])
midpoint = 0.5 * point_a + 0.5 * point_b
wrong_ok = in_annulus(point_a) and in_annulus(point_b) and in_annulus(midpoint)
box_projection = np.clip(midpoint, -1.0, 1.0)
box_feasible = np.all((-1.0 <= box_projection) & (box_projection <= 1.0))
print("annulus midpoint feasible:", in_annulus(midpoint))
print("connected-implies-convex test would be:", wrong_ok)
print("convex box projection feasible:", box_feasible)

## Evaluate it + Practice

- Metric: final loss; compare against a no-skill baseline that keeps the initial point or takes a fixed tiny step.
- Sanity check: on D1, verify the output against the closed-form minimizer or exact linear-system solution.
- Ablation: turn off the key safeguard or curvature idea and confirm the metric worsens.
- Failure signals: nondecreasing loss, nonpositive CG denominator, exploding path length, or a tiny gradient with bad curvature.
- Reproducibility: keep the seed fixed and do not download data; `load_breast_cancer` is bundled with sklearn.

Practice prompts:

1. Change one tolerance and predict which rung changes the most.

2. Replace D2's condition number and compare the metric curve.

3. Add one diagnostic print that would catch the pitfall earlier.